# OmniVoice — Server GPU trên Colab

Tổng hợp giọng nói chạy trên GPU Colab, máy ở nhà gọi lên qua API.

**Chỉ có client/server.** Không giao diện, không quản lý dự án — đúng ba việc:
nhiều worker INT4, hàng đợi, trả kết quả về client.

## Chạy lần lượt từ trên xuống

| ô | việc | lần đầu | lần sau |
|---|---|---|---|
| 1 | kiểm tra GPU | vài giây | vài giây |
| 2 | clone repo + lấy runtime C++ | **40–90 phút** nếu phải build (Colab free chỉ 2 vCPU) | **vài giây** nếu đã có bản dựng sẵn |
| 3 | tải model INT4 (~660 MB) | 1–2 phút | vài giây |
| 4 | bật server | ~30 giây | ~30 giây |
| 5 | mở đường hầm, lấy URL + key | ~20 giây | ~20 giây |
| 7 | giữ phiên sống, tự dựng lại khi hỏng | chạy liên tục | chạy liên tục |

Xong ô 5 thì chép URL với API key về máy mình rồi chạy `remote/client.py`.

## Số worker đặt bao nhiêu

Số đo thật trên RTX 4000 Ada (`examples/bench_parallel.py`):

| worker | tăng tốc | VRAM | audio so với 1 luồng |
|---|---|---|---|
| 1 | 1.00x | 1443 MiB | mốc chuẩn |
| 2 | 1.80x | 2452 MiB | giống hệt từng byte |
| **4** | **2.60x** | 4802 MiB | giống hệt từng byte |
| 6 | 0.99x | 7140 MiB | giống hệt từng byte |
| 8 | 0.94x | 9440 MiB | giống hệt từng byte |

**4 là điểm tối ưu** — quá 4 thì chậm đi chứ không nhanh thêm, GPU đã bão hoà.
Chia luồng **không đổi chất lượng**: băm SHA1 nội dung audio khớp 100% với bản
chạy tuần tự cùng seed. T4 của Colab yếu hơn nên tốc độ tuyệt đối thấp hơn,
nhưng hình dạng đường cong giữ nguyên. Server tự hạ số worker nếu VRAM không đủ.

## Cần biết trước

- Phiên Colab tự ngắt sau vài giờ, mọi thứ trong `/content` mất theo. Bật
  `USE_DRIVE_CACHE` ở ô 2 thì lần sau khỏi build lại.
- URL cloudflared đổi mỗi lần chạy lại ô 5.
- Server chỉ chạy CUDA, không có đường lui về CPU — không có GPU là dừng ngay.


In [ ]:
# ── 1. Kiểm tra GPU ─────────────────────────────────────────────────────────
import shutil, subprocess

if not shutil.which("nvidia-smi"):
    raise SystemExit(
        "Runtime nay KHONG co GPU.\n"
        "Sua: menu Runtime -> Change runtime type -> T4 GPU, roi chay lai tu dau."
    )

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
cc = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                    capture_output=True, text=True).stdout.strip().splitlines()[0]
CUDA_ARCH = cc.replace(".", "")
print(f"compute capability {cc} -> build rieng cho sm_{CUDA_ARCH}, nhanh hon build da kien truc")


In [ ]:
# ── 2. Lấy runtime C++ ──────────────────────────────────────────────────────
# Thu tu: co san -> Drive -> tai ban build san -> build tu nguon.
# Build tu nguon mat 40-90 phut tren Colab free (chi 2 vCPU) nen chi lam MOT
# LAN roi dong goi len GitHub Release; cac phien sau chi tai ve vai giay.
import os, shutil, subprocess, urllib.request
from pathlib import Path

REPO_URL   = "https://github.com/GrayPham/submodulevoice.git"
BRANCH     = "master"
GIT_TOKEN  = ""        # chi can khi repo dat private
USE_DRIVE_CACHE = False
DRIVE_CACHE = "/content/drive/MyDrive/omnivoice-build"

PREBUILT = (REPO_URL.replace(".git", "")
            + f"/releases/download/runtime-linux/omnivoice-linux-cuda-sm{CUDA_ARCH}.tar.gz")

APP   = Path("/content/submodulevoice")
SRC   = Path("/content/omnivoice.cpp")
BUILD = SRC / "build"
LIB   = BUILD / "libomnivoice.so"

# Chay lenh va IN TUNG DONG ra ngay. subprocess.run ghi thang ra file
# descriptor cua OS, ma Colab chi bat sys.stdout o tang Python -> khong thay
# gi cho den khi ca o chay xong. Doc qua pipe roi print lai thi tien do hien
# ngay. tail=1: bot bot dong cho do ngop, build sinh hang nghin dong.
def sh(cmd, cwd=None, tail=0):
    print("$", cmd, flush=True)
    pr = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True,
                          encoding="utf-8", errors="replace", bufsize=1)
    n = 0
    for line in pr.stdout:
        n += 1
        line = line.rstrip()
        # cmake --build in dang "[ 42%] Building ..." -> chi giu dong co phan tram
        # hoac dong loi, con lai bo bot cho do ngop.
        if tail and not (line.startswith("[") or "rror" in line or "arning: " in line):
            if n % 50:
                continue
        print(" ", line, flush=True)
    pr.wait()
    if pr.returncode:
        raise SystemExit(f"that bai (ma {pr.returncode}): {cmd}")

if USE_DRIVE_CACHE:
    from google.colab import drive; drive.mount("/content/drive")

url = REPO_URL
if GIT_TOKEN:
    url = REPO_URL.replace("https://", f"https://{GIT_TOKEN}@")
if APP.exists():
    sh(f"git -C {APP} fetch --depth 1 origin {BRANCH} && "
       f"git -C {APP} reset --hard origin/{BRANCH}")
else:
    sh(f"git clone --depth 1 -b {BRANCH} {url} {APP}")
sh("pip install -q numpy huggingface_hub")

def try_prebuilt() -> bool:
    tgz = "/content/runtime.tar.gz"
    try:
        print(f"thu tai ban build san cho sm_{CUDA_ARCH} ...", flush=True)
        urllib.request.urlretrieve(PREBUILT, tgz)
    except Exception as e:
        print(f"  chua co ban build san ({type(e).__name__}) -> build tu nguon")
        return False
    BUILD.mkdir(parents=True, exist_ok=True)
    sh(f"tar xzf {tgz} -C {BUILD}")
    if LIB.exists():
        print("  dung ban build san, bo qua buoc build.")
        return True
    print("  goi tai ve sai dinh dang -> build tu nguon")
    return False

if LIB.exists():
    print("da co ban build trong /content.")
elif USE_DRIVE_CACHE and Path(DRIVE_CACHE, "libomnivoice.so").exists():
    print("khoi phuc build tu Drive ...")
    BUILD.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_CACHE, BUILD, dirs_exist_ok=True)
elif not try_prebuilt():
    if not SRC.exists():
        sh("git clone --recurse-submodules --depth 1 "
           "https://github.com/ServeurpersoCom/omnivoice.cpp.git /content/omnivoice.cpp")
    sh(f"cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON "
       f"-DOMNIVOICE_SHARED=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}", cwd=SRC)
    print(f"build bang $(nproc) nhan — Colab free thuong chi co 2, "
      f"nen buoc nay co the mat 40-90 phut. Chi phai lam MOT LAN.", flush=True)
    sh("nproc && cmake --build build -j$(nproc)", cwd=SRC, tail=1)

    # Linux dat ten thu vien co phien ban: file that la libggml-base.so.0.17.0,
    # con .so va .so.0 chi la symlink. Glob "*.so" chi lay symlink -> goi ra
    # mot dong symlink tro vao hu khong, va loi chi lo o phien sau.
    need = ["libomnivoice.so", "libggml.so", "libggml-base.so",
            "libggml-cpu.so", "libggml-cuda.so"]
    have = {f.name for f in BUILD.rglob("*.so*")}
    thieu = [x for x in need if x not in have]
    assert not thieu, f"thieu thu vien: {thieu}"
    tgz = f"/content/omnivoice-linux-cuda-sm{CUDA_ARCH}.tar.gz"
    sh(f"cd {BUILD} && find . -name '*.so*' -print0 | tar czf {tgz} --null -T -")
    mb = Path(tgz).stat().st_size / 2**20
    print("=" * 72)
    print(f"  DA DONG GOI: {tgz}  ({mb:.0f} MB)")
    print("  Tai file nay ve may (bang File ben trai), roi tren GitHub:")
    print("    Releases -> Draft a new release -> tag: runtime-linux")
    print("    -> dinh kem file tren -> Publish")
    print("  Tu lan sau notebook tu tai ve, khong phai build lai 15 phut.")
    print("=" * 72)
    if USE_DRIVE_CACHE:
        Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)
        for f in BUILD.rglob("*.so*"):
            shutil.copy2(f, Path(DRIVE_CACHE, f.name))
        print("da luu build vao Drive.")

assert LIB.exists(), "khong thay libomnivoice.so"
os.environ["OMNIVOICE_LIB"] = str(BUILD)
print("thu vien:", LIB)
print(sorted(p.name for p in BUILD.rglob("*.so*")))


In [ ]:
# ── 3. Tải model INT4 (~660 MB) ─────────────────────────────────────────────
from pathlib import Path
from huggingface_hub import hf_hub_download

MODELS = Path("/content/models"); MODELS.mkdir(exist_ok=True)
for f in ["omnivoice-base-Q4_K_M.gguf", "omnivoice-tokenizer-Q8_0.gguf"]:
    if (MODELS / f).exists():
        print("[co san]", f)
    else:
        print("[tai]", f, flush=True)
        hf_hub_download("Serveurperso/OmniVoice-GGUF", f, local_dir=str(MODELS))
print(sorted(p.name for p in MODELS.glob("*.gguf")))


In [ ]:
# ── 4. Bật server ───────────────────────────────────────────────────────────
import json, os, secrets, subprocess, time, urllib.request

PORT    = 8770
WORKERS = 4          # diem toi uu do duoc; server tu ha neu VRAM khong du
API_KEY = secrets.token_urlsafe(12)
LOG     = "/content/server.log"

subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
time.sleep(1)

env = dict(os.environ, OMNIVOICE_LIB="/content/omnivoice.cpp/build", PYTHONUNBUFFERED="1")
with open(LOG, "wb") as f:
    subprocess.Popen(
        ["python", "remote/server.py", "--workers", str(WORKERS), "--port", str(PORT),
         "--key", API_KEY, "--models-dir", "/content/models", "--profile", "lite"],
        stdout=f, stderr=subprocess.STDOUT, env=env, cwd="/content/submodulevoice")

for _ in range(180):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3) as r:
            h = json.load(r)
        print(json.dumps(h, ensure_ascii=False, indent=2))
        print(f"\nSERVER SAN SANG - {h['workers']} worker")
        break
    except Exception:
        time.sleep(2)
else:
    print(open(LOG, encoding="utf-8", errors="replace").read()[-4000:])
    raise SystemExit("server khong len, xem log o tren.")


In [ ]:
# ── 5. Mở đường hầm, lấy URL + API key ──────────────────────────────────────
import re, subprocess, time
from pathlib import Path

if not Path("/content/cloudflared").exists():
    subprocess.run(
        "wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/"
        "releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared",
        shell=True, check=True)

subprocess.run("pkill -f cloudflared || true", shell=True)
subprocess.Popen(f"/content/cloudflared tunnel --url http://127.0.0.1:{PORT} "
                 f"--no-autoupdate > /content/cloudflared.log 2>&1", shell=True)

URL = None
for _ in range(40):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                  Path("/content/cloudflared.log").read_text(errors="replace"))
    if m:
        URL = m.group(0); break

if not URL:
    print(Path("/content/cloudflared.log").read_text(errors="replace")[-3000:])
    raise SystemExit("khong lay duoc URL duong ham")

print("=" * 72)
print("  CHEP VE MAY MINH")
print("=" * 72)
print(f"  URL      {URL}")
print(f"  API key  {API_KEY}")
print("=" * 72)
print("\nLenh chay o may minh:\n")
print(f"  python remote/client.py --url {URL} --key {API_KEY} \\")
print( "      --script scripts/kichban_pt.txt \\")
print( "      --ref output/refs3/FDown.vn_Tai_video_Facebook_MP3_9995-ref.wav \\")
print( "      --lang Portuguese --concurrency 4 -o output/remote/ket-qua.wav")


In [ ]:
# ── 6. Tự kiểm tra ──────────────────────────────────────────────────────────
# Thu HAI buoc rieng biet. Gop lam mot thi khi hong khong biet loi o server
# hay o duong ham — dung loi tung gap: gaierror khi DNS khong tra duoc ten
# mien trycloudflare, trong khi server van chay tot.
import json, time, urllib.request
from IPython.display import Audio, display

PAYLOAD = json.dumps({"text": "Xin chao, day la bai kiem tra ket noi.",
                      "lang": "Vietnamese", "steps": 16}).encode()
HDRS = {"Content-Type": "application/json", "X-API-Key": API_KEY}

def call(base, timeout=300):
    t0 = time.perf_counter()
    req = urllib.request.Request(base + "/tts", data=PAYLOAD, headers=HDRS, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        wav = r.read()
        synth = float(r.headers.get("X-Synth-Seconds", 0))
        audio = float(r.headers.get("X-Audio-Seconds", 0))
    return wav, synth, audio, time.perf_counter() - t0

# --- buoc 1: server, khong qua mang ngoai ------------------------------------
print("[1/2] goi thang 127.0.0.1 ...", flush=True)
try:
    wav, synth, audio, rtt = call(f"http://127.0.0.1:{PORT}")
    print(f"  OK — audio {audio:.2f}s, GPU tinh {synth:.2f}s")
    open("/content/test.wav", "wb").write(wav)
    display(Audio("/content/test.wav"))
    server_ok = True
except Exception as e:
    server_ok = False
    print(f"  HONG: {type(e).__name__}: {e}")
    print("  -> loi o SERVER, khong phai duong ham. Xem log:")
    print(open(LOG, encoding="utf-8", errors="replace").read()[-3000:])

# --- buoc 2: duong ham -------------------------------------------------------
if server_ok:
    print(f"\n[2/2] goi qua duong ham {URL} ...", flush=True)
    try:
        _w, s2, a2, rtt2 = call(URL)
        print(f"  OK — khu hoi {rtt2:.2f}s, trong do GPU {s2:.2f}s, "
              f"phan mang {rtt2 - s2:.2f}s")
    except Exception as e:
        print(f"  HONG: {type(e).__name__}: {e}")
        print("  Server VAN TOT, chi duong ham co van de. Cach xu ly:")
        print("    - chay lai o 5 de lay URL moi")
        print("    - kiem tra: !pgrep -f cloudflared  va  !tail -25 /content/cloudflared.log")
        print("    - loi 'Name or service not known' = DNS chua tra duoc ten mien,")
        print("      thuong do tunnel vua chet hoac chua kip lan truyen; doi 30s roi thu lai")


In [ ]:
# ── 7. Giữ phiên sống + tự dựng lại khi hỏng (để ô này CHẠY LIÊN TỤC) ───────
#
# Colab thu hồi runtime khi thấy TRÌNH DUYỆT không hoạt động, chứ không nhìn
# GPU có bận hay không. Một ô đang chạy được tính là hoạt động, nên vòng lặp
# này vừa giữ phiên vừa làm việc thật: theo dõi server, dựng lại nếu nó chết,
# mở lại đường hầm nếu URL rơi.
#
# Không tránh được giới hạn cứng (~12h free, ~24h Pro). Hết là hết.
# Ctrl+C hoặc bấm nút dừng để thoát.
import json, subprocess, time, urllib.request
from datetime import datetime
from pathlib import Path

CHECK_EVERY = 60          # giay
last_served = -1

def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

def tunnel_alive() -> bool:
    return subprocess.run("pgrep -f cloudflared", shell=True,
                          capture_output=True).returncode == 0

print(f"theo doi moi {CHECK_EVERY}s. De o nay chay lien tuc.\n")
while True:
    h = health()
    ts = datetime.now().strftime("%H:%M:%S")

    if h is None:
        print(f"[{ts}] SERVER CHET -> dung lai ...", flush=True)
        with open(LOG, "ab") as f:
            subprocess.Popen(
                ["python", "remote/server.py", "--workers", str(WORKERS),
                 "--port", str(PORT), "--key", API_KEY,
                 "--models-dir", "/content/models", "--profile", "lite"],
                stdout=f, stderr=subprocess.STDOUT,
                env=dict(os.environ, OMNIVOICE_LIB="/content/omnivoice.cpp/build",
                         PYTHONUNBUFFERED="1"),
                cwd="/content/submodulevoice")
        time.sleep(45)
        print(f"[{ts}] {'da len lai' if health() else 'VAN CHUA LEN, xem ' + LOG}", flush=True)
    else:
        d = h["served"] - last_served if last_served >= 0 else h["served"]
        last_served = h["served"]
        gb = h.get("gpu", {})
        print(f"[{ts}] worker {h['busy']}/{h['workers']} ban | doi {h['queued']} | "
              f"da xong {h['served']} (+{d}) | loi {h['failed']} | "
              f"VRAM trong {gb.get('vram_free_mib')} MiB", flush=True)

    if not tunnel_alive():
        print(f"[{ts}] DUONG HAM RO'I -> mo lai ...", flush=True)
        subprocess.Popen(f"/content/cloudflared tunnel --url http://127.0.0.1:{PORT} "
                         f"--no-autoupdate > /content/cloudflared.log 2>&1", shell=True)
        time.sleep(20)
        import re as _re
        m = _re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                       Path("/content/cloudflared.log").read_text(errors="replace"))
        if m and m.group(0) != URL:
            URL = m.group(0)
            print(f"[{ts}] URL MOI: {URL}")
            print(f"        chay lai client voi URL nay, no se lam tiep phan con thieu")

    time.sleep(CHECK_EVERY)
